# Deep Dive: Structured Outputs & Context Engineering

## 📋 Summary

Free-form prose breaks downstream code, and dumping raw 50-field JSON payloads and 48-turn transcripts into a prompt causes **context rot** and **"lost in the middle"** accuracy drops.

In this notebook (covering **Lesson 07**: Structured Outputs with Zod/Pydantic + the 4-stage Python Context Engineering progression), you will build a **Real Estate Lease & Property Inspection Pipeline** demonstrating:
1. **Guaranteed Structured Outputs** via Pydantic → JSON Schema + forced `tool_choice`
2. **Stage 1 — Deterministic Tool-Output Pruning** (filtering bloated API payloads to decision-load-bearing fields)
3. **Stage 2 — Bounded Typed Fact Block** (pinning critical facts under a strict token cap)
4. **Stage 3 — Budgeted Compression** of resolved conversation topics
5. **Stage 4 — Position-Aware Context Assembly** (placing facts at the top boundary, resolved summaries in the middle, and active issues verbatim at the bottom boundary)

### Key concepts in this notebook

| # | Concept | Complexity |
|---|---|---|
| 1 | Why prompt-only JSON fails vs schema-enforced tool output | 🟢 Beginner |
| 2 | Pydantic models → JSON Schema + `tool_choice` extraction | 🟢 Beginner |
| 3 | Deterministic tool-output pruning (token reduction without LLM calls) | 🟡 Intermediate |
| 4 | Bounded Fact Blocks & budgeted topic compression | 🟡 Intermediate |
| 5 | Position-aware context assembly ("lost in the middle" mitigation) | 🔴 Advanced |
| 6 | Hands-on schema & context engineering exercises | 🏋️ Practice |

> **How to use this notebook:** Run cells in order. Sections 3–5 include offline deterministic steps plus live Claude calls. Cells with `# TODO` are exercises.

## 0. Setup

In [ ]:
%pip install -q anthropic pydantic

import json
import os
from getpass import getpass
from typing import Literal
from pydantic import BaseModel, Field

try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    if not os.environ.get("ANTHROPIC_API_KEY"):
        os.environ["ANTHROPIC_API_KEY"] = getpass("Enter your ANTHROPIC_API_KEY: ")

import anthropic

client = anthropic.Anthropic()
MODEL = "claude-haiku-4-5-20251001"


def approx_tokens(text: str) -> int:
    """Fast deterministic token estimator (~4 chars/token) for offline budgeting."""
    return max(1, len(text) // 4)


print("Setup complete. Model:", MODEL)

## 1. Guaranteed Structured Outputs with Pydantic + `tool_choice` 🟢

In TypeScript (Lesson 07), you define a **Zod** schema and convert it with `zodToJsonSchema`. In Python, **Pydantic** (`BaseModel.model_json_schema()`) plays the exact same role.

By passing that schema as a tool and forcing `tool_choice={"type": "tool", "name": "..."}`, Claude is constrained to return a JSON object matching your schema — no markdown fences, no prose preamble, and validated immediately by Pydantic.

In [ ]:
class InspectionFinding(BaseModel):
    area: Literal["roof", "plumbing", "electrical", "hvac", "foundation"]
    severity: Literal["low", "medium", "high", "critical"]
    summary: str = Field(..., max_length=160, description="One-sentence summary of the defect")
    estimated_repair_usd: int = Field(..., ge=0, description="Estimated repair cost in USD")


class PropertyInspectionReport(BaseModel):
    property_id: str
    overall_rating: Literal["pass", "conditional", "fail"]
    findings: list[InspectionFinding] = Field(..., min_length=1)
    safe_to_close: bool


def extract_structured(prompt: str, schema_cls: type[BaseModel], tool_name: str = "record_output") -> BaseModel:
    resp = client.messages.create(
        model=MODEL,
        max_tokens=1024,
        tools=[{
            "name": tool_name,
            "description": f"Return structured data matching {schema_cls.__name__}.",
            "input_schema": schema_cls.model_json_schema(),
        }],
        tool_choice={"type": "tool", "name": tool_name},
        messages=[{"role": "user", "content": prompt}],
    )
    tool_block = next(b for b in resp.content if b.type == "tool_use")
    return schema_cls.model_validate(tool_block.input)


raw_notes = """
Inspector notes for PROP-882 (412 Oak Ave):
- Main electrical panel is an outdated Federal Pacific Stab-Lok unit; serious fire hazard, replace ASAP (~$3,800).
- Minor drip under guest bathroom sink P-trap; quick fix (~$150).
- Roof shingles have ~12 years remaining, no leaks.
Recommend holding closing until electrical panel is replaced.
"""

report = extract_structured(f"Extract an inspection report from:\n{raw_notes}", PropertyInspectionReport)
print(report.model_dump_json(indent=2))

### 🧐 Observation
- Notice `model_validate(tool_block.input)` guarantees both types (`int`, `bool`) and constraints (`Literal`, `ge=0`, `max_length=160`).
- Contrast this with asking *"Please reply in JSON"* — prompt-only JSON frequently wraps output in triple backticks or drifts on enum casing.

## 2. Context Engineering Stage 1: Deterministic Tool-Output Pruning 🟡

In Lesson 07's Python module (`01-prune-tool-output`), a raw `lookup_order` API payload had **57 fields**, of which only **5** actually mattered for refund eligibility. Dumping all 57 fields wastes tokens and distracts attention.

**Rule:** Prune verbose tool outputs **deterministically in application code** *before* inserting them into the LLM context.

In [ ]:
RAW_LEASE_API_RESPONSE = {
    "lease_id": "LSE-2026-041",
    "tenant_name": "Jordan Lee",
    "monthly_rent_usd": 2450,
    "security_deposit_usd": 4900,
    "lease_end_date": "2026-06-30",
    "notice_period_days": 60,
    # 18 noisy metadata/audit fields irrelevant to renewal decisions:
    "crm_sync_uuid": "9d8a7f6e-5c4b-3a21-90ef-123456789abc",
    "legacy_erp_code": "ERP-99104-B",
    "marketing_attribution": {"utm_source": "zillow", "utm_campaign": "spring25", "click_id": "zk_981234"},
    "internal_audit_trail": [{"ts": f"2025-07-0{i}T12:00:00Z", "actor": "bot", "op": "sync"} for i in range(1, 9)],
    "pdf_sha256": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",
    "billing_gateway_token": "tok_redacted_00019283",
}

KEEP_FIELDS = ("lease_id", "monthly_rent_usd", "security_deposit_usd", "lease_end_date", "notice_period_days")


def prune_lease_payload(raw: dict) -> dict:
    return {k: raw[k] for k in KEEP_FIELDS if k in raw}


raw_json = json.dumps(RAW_LEASE_API_RESPONSE)
pruned_json = json.dumps(prune_lease_payload(RAW_LEASE_API_RESPONSE))

print(f"Raw payload:    {len(raw_json):4d} chars (~{approx_tokens(raw_json)} tokens)")
print(f"Pruned payload: {len(pruned_json):4d} chars (~{approx_tokens(pruned_json)} tokens)")
print(f"Reduction:      {(1 - len(pruned_json)/len(raw_json))*100:.1f}%")
print("Pruned:", pruned_json)

### 🧐 Observation
- Deterministic pruning costs **$0** and **0 ms** of LLM latency, removes sensitive/distracting tokens (like `billing_gateway_token`), and cuts context size by >75%.

## 3. Stages 2 & 3: Bounded Fact Blocks & Budgeted Compression 🟡

As a multi-turn conversation grows across multiple topics (e.g., first discussing a parking spot, then a pet addendum, and *now* a lease renewal):
- **Stage 2 (Case Facts Block):** Extract immutable, load-bearing facts into a short, structured block under a hard token ceiling (e.g., `<= 250` tokens).
- **Stage 3 (Budgeted Compression):** Summarize *resolved* sub-topics into compact narratives with explicit token budgets, while keeping the *active* sub-topic **byte-exact verbatim** so no current detail is lost.

In [ ]:
def build_fact_block(pruned_lease: dict, max_tokens: int = 200) -> str:
    lines = ["# Property & Lease Facts"] + [f"- **{k}**: {v}" for k, v in pruned_lease.items()]
    block = "\n".join(lines)
    assert approx_tokens(block) <= max_tokens, f"Fact block exceeded {max_tokens} tokens"
    return block


def compress_resolved_topic(topic_title: str, transcript: str, max_output_tokens: int = 120) -> str:
    resp = client.messages.create(
        model=MODEL,
        max_tokens=max_output_tokens,
        system="Compress this resolved conversation into <= 3 bullet points preserving only final decisions, dates, and dollar amounts.",
        messages=[{"role": "user", "content": transcript}],
    )
    return f"# Resolved: {topic_title}\n{resp.content[0].text.strip()}"


resolved_parking = (
    "Tenant: Can I switch from spot B-12 to a covered spot?\n"
    "Agent: Covered spot C-04 is available for +$75/mo starting May 1.\n"
    "Tenant: Great, please assign C-04 starting May 1.\n"
    "Agent: Confirmed! Spot C-04 assigned effective May 1, +$75/mo."
)

fact_block = build_fact_block(prune_lease_payload(RAW_LEASE_API_RESPONSE))
compressed_parking = compress_resolved_topic("Parking reassignment", resolved_parking)
print(fact_block)
print("\n" + compressed_parking)

## 4. Stage 4: Position-Aware Context Assembly 🔴

LLMs attend most strongly to the **beginning** and **end** of the prompt ("lost in the middle"). Lesson 07's `assemble.py` uses a strict top-to-bottom layout:

```
┌────────────────────────────────────────────────────────┐
│ TOP BOUNDARY    │ # Case / Lease Facts (structured)    │ ← High attention
├────────────────────────────────────────────────────────┤
│ MIDDLE ZONE     │ # Resolved topics (compressed)       │ ← Lower attention (OK!)
├────────────────────────────────────────────────────────┤
│ BOTTOM BOUNDARY │ # Active issue (byte-exact verbatim) │ ← High attention, adjacent to new turn
└────────────────────────────────────────────────────────┘
```

In [ ]:
def assemble_context(
    fact_block: str,
    resolved_blocks: list[str],
    active_title: str,
    active_verbatim: str,
) -> str:
    """Position-aware assembly: facts at top, resolved in middle, active verbatim at bottom."""
    sections = [
        fact_block.strip(),
        *[b.strip() for b in resolved_blocks],
        f"# Active issue: {active_title}\n{active_verbatim.strip()}",
    ]
    return "\n\n---\n\n".join(sections)


active_renewal_turns = (
    "Tenant: If I want to move out on June 30, what is the exact last day I can give notice?\n"
    "Tenant: Also, will my $4,900 deposit cover the last month if I miss the notice window?"
)

assembled = assemble_context(
    fact_block=fact_block,
    resolved_blocks=[compressed_parking],
    active_title="Lease termination notice & deposit",
    active_verbatim=active_renewal_turns,
)

print(assembled)
print(f"\nTotal assembled context: ~{approx_tokens(assembled)} tokens")

### Failure Modes & Fixes

| Failure Mode | Why It Happens | Fix |
|---|---|---|
| JSON parse failures in production | Relying on prompt text instead of schema-enforced tool output | Use Pydantic/Zod → `input_schema` + `tool_choice` + `model_validate()` |
| Token explosion & distracted reasoning | Dumping raw 50+ field API responses into messages | Deterministically prune tool payloads to decision-critical fields |
| Losing active user nuances | Summarizing the *current* open topic too aggressively | Keep the active issue **byte-exact verbatim**; only compress *resolved* topics |
| Key facts ignored mid-conversation | Burying structured facts in the middle of a 40-turn transcript | Pin structured facts at the top boundary and active turns at the bottom |

## 5. Exercises 🏋️

### Exercise 1: Pydantic Schema Extraction for Tenant Maintenance Requests
Define a Pydantic model `MaintenanceTicket` (`unit: str`, `category: Literal[...]`, `urgency: Literal["routine", "urgent", "emergency"]`, `requires_entry_permission: bool`, `summary: str`) and test `extract_structured(...)` on two sample messages.

In [ ]:
class MaintenanceTicket(BaseModel):
    # TODO: Define fields with Literal types and Field constraints
    summary: str = Field(..., description="Short summary")


sample_requests = [
    "Unit 3B here — water is pouring through the ceiling from upstairs! Come in immediately!",
    "Unit 12A: bedroom closet door is squeaking. No rush, please email before entering.",
]

# TODO: Call extract_structured for each request and print validated JSON
for msg in sample_requests:
    print("Ready to extract:", msg[:40])

### Exercise 2: Write a Context Assembly Contract Test
In Lesson 07 (`test_assemble.py`), unit tests verify **section ordering** and **byte-exact preservation** of the active issue. Write a function `verify_assembly_contract(assembled: str, active_verbatim: str)` that asserts:
1. `# Property & Lease Facts` appears before `# Resolved:`
2. `# Resolved:` appears before `# Active issue:`
3. `active_verbatim.strip()` appears **byte-for-byte** inside `assembled`

In [ ]:
def verify_assembly_contract(assembled_md: str, active_verbatim: str) -> bool:
    """Raise AssertionError if section ordering or verbatim preservation is violated."""
    # TODO: Compute index positions for the 3 headers and assert strict ordering
    # TODO: Assert active_verbatim.strip() in assembled_md
    return False


# Test against `assembled` from Section 4
print("Contract passed:", verify_assembly_contract(assembled, active_renewal_turns))

## 🏁 Summary & Key Takeaways

- **Schema-first outputs:** Pair Pydantic/Zod schemas with forced `tool_choice` and `model_validate()` so downstream code never parses fragile prose.
- **Prune before the prompt:** Strip noisy API metadata deterministically in Python before the LLM ever sees it.
- **Engineer context positionally:** Pin compact typed facts at the top, compress resolved history in the middle, and keep the active issue verbatim at the bottom.

### What's next?
Continue to **`deep-dive-5-claude-skills-and-mcp-integration.ipynb`** to package domain expertise into reusable Claude Skills (`SKILL.md`) and connect external tools via the Model Context Protocol (MCP).